# Phase 0 — ACNE04 Dataset Analysis

**AI Skincare Assistant · Dataset Investigation & Validation**

> This notebook is **presentational only**. All business logic lives in `phase0/src/`.
> Run the pipeline first: `python phase0/scripts/run_all.py`
>
> Then execute cells top-to-bottom to load pre-computed artifacts and display results.

---
**Dataset**: ACNE04 (Kaggle distribution, 1024×1024 px)  
**Severity labels** (Hayashi grading): `0=Mild · 1=Moderate · 2=Severe · 3=Very Severe`  
**Note on `sim_acne.csv`**: This file is a Kaggle-uploader synthetic artefact for N-of-1 trial simulation — it is NOT part of the ACNE04 dataset and is excluded from all analysis.

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Markdown, Image

# ── Resolve paths ────────────────────────────────────────────────────────────
NOTEBOOK_DIR  = Path().resolve()
PHASE0_ROOT   = NOTEBOOK_DIR.parent          # phase0/
PROJECT_ROOT  = PHASE0_ROOT.parent           # skincare-ai/
SRC_ROOT      = PHASE0_ROOT / 'src'

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

OUTPUTS    = PROJECT_ROOT / 'data' / 'phase0_outputs'
FIGURES    = PROJECT_ROOT / 'reports' / 'phase0' / 'figures'
REPORTS    = OUTPUTS / 'reports'

print(f'Project root : {PROJECT_ROOT}')
print(f'Outputs      : {OUTPUTS}')
print(f'Figures      : {FIGURES}')

In [ ]:
def load_json(path: Path) -> dict:
    """Load JSON; return empty dict with warning if missing."""
    if not path.exists():
        print(f'⚠️  Not found: {path.name}  —  run the pipeline first.')
        return {}
    with open(path) as f:
        return json.load(f)

def load_csv(path: Path) -> pd.DataFrame:
    """Load CSV; return empty DataFrame with warning if missing."""
    if not path.exists():
        print(f'⚠️  Not found: {path.name}  —  run the pipeline first.')
        return pd.DataFrame()
    return pd.read_csv(path)

def show_figure(name: str, caption: str = '') -> None:
    """Display a saved figure from the figures directory."""
    path = FIGURES / name
    if not path.exists():
        print(f'⚠️  Figure not found: {name}')
        return
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(mpimg.imread(str(path)))
    ax.axis('off')
    if caption:
        ax.set_title(caption, fontsize=12, pad=8)
    plt.tight_layout()
    plt.show()

print('Helpers loaded.')

---
## 1 · Dataset Inventory

In [ ]:
ingestion_log = load_json(OUTPUTS / 'ingestion_log.json')

if ingestion_log:
    display(Markdown(f"""
| Metric | Value |
|--------|-------|
| Format detected | `{ingestion_log.get('format_detected', '—')}` |
| Total images (manifest) | **{ingestion_log.get('total_records', '—'):,}** |
| Duplicate images removed | {ingestion_log.get('duplicate_count', '—')} |
| Images not on disk | {ingestion_log.get('images_not_found', '—')} |
| `sim_acne.csv` status | Excluded (N-of-1 synthetic artefact) |
"""))

    disc = ingestion_log.get('all_folder_discrepancy', {})
    if disc:
        display(Markdown(f"""
### `all_1024` Discrepancy Resolution
| Status | Count |
|--------|-------|
| In class folder | {disc.get('in_class_folder', '—')} |
| Unlabelled (not in any class folder) | {disc.get('unlabelled', '—')} |
| Duplicate within all_1024 | {disc.get('duplicate_in_all', '—')} |
"""))

In [ ]:
manifest = load_csv(OUTPUTS / 'manifest.csv')
if not manifest.empty:
    display(manifest.groupby(['severity_label', 'severity_name']).agg(
        count=('image_id', 'count'),
        duplicates=('is_duplicate', 'sum')
    ).reset_index())

---
## 2 · Class Distribution & Imbalance

In [ ]:
eda_stats = load_json(OUTPUTS / 'eda_stats.json')

if eda_stats:
    dist = eda_stats.get('class_distribution', {})
    ir   = eda_stats.get('imbalance_ratio', None)
    wts  = eda_stats.get('recommended_weights', {})

    display(Markdown(f'**Imbalance Ratio (IR)** = `{ir:.2f}` — threshold is 10.0' if ir else ''))
    display(Markdown('> IR > 5 → `WeightedRandomSampler` + class-weighted loss is **mandatory** in Phase 1.'))

    if dist:
        rows = []
        total = sum(dist.values())
        for name, count in dist.items():
            rows.append({'Severity': name, 'Count': count,
                         'Pct': f"{count/total*100:.1f}%",
                         'Inv-freq weight': f"{wts.get(name, '—'):.3f}"})
        display(pd.DataFrame(rows))

In [ ]:
show_figure('01_class_distribution.png', 'Class Distribution (Hayashi grades 0–3)')

In [ ]:
show_figure('01_all_folder_discrepancy.png', 'all_1024 vs Class Folders — Discrepancy Analysis')

---
## 3 · Image Resolution

In [ ]:
if eda_stats:
    res = eda_stats.get('resolution', {})
    if res:
        display(Markdown(f"""
| Stat | Width | Height |
|------|-------|--------|
| Min | {res.get('width_stats',{}).get('min','—')} | {res.get('height_stats',{}).get('min','—')} |
| Max | {res.get('width_stats',{}).get('max','—')} | {res.get('height_stats',{}).get('max','—')} |
| Mean | {res.get('width_stats',{}).get('mean','—'):.1f} | {res.get('height_stats',{}).get('mean','—'):.1f} |
| Images below 224px | {res.get('n_below_min_dimension','—')} | — |
"""))

show_figure('02_resolution_scatter.png', 'Resolution Scatter (coloured by severity)')

---
## 4 · Sample Images per Class

In [ ]:
for severity_name in ['mild', 'moderate', 'severe', 'very_severe']:
    show_figure(f'02_sample_grid_{severity_name}.png',
                f'Sample Grid — {severity_name.replace("_", " ").title()}')

---
## 5 · Image Quality Audit

In [ ]:
qa = load_csv(OUTPUTS / 'quality_audit.csv')

if not qa.empty:
    total = len(qa)
    qp    = qa['quality_pass'].sum()

    display(Markdown(f"""
| Metric | Count | % |
|--------|-------|---|
| Total images audited | {total} | 100% |
| **Quality pass** | **{qp}** | **{qp/total*100:.1f}%** |
| Corrupted | {qa['is_corrupted'].sum()} | {qa['is_corrupted'].mean()*100:.1f}% |
| Blurry | {qa['is_blurry'].sum()} | {qa['is_blurry'].mean()*100:.1f}% |
| No face detected | {(qa['face_flag']=='no_face').sum()} | {(qa['face_flag']=='no_face').mean()*100:.1f}% |
| Multiple faces | {(qa['face_flag']=='multi_face').sum()} | {(qa['face_flag']=='multi_face').mean()*100:.1f}% |
"""))

    # Per-class quality pass rate
    display(qa.groupby('severity_name')['quality_pass'].agg(['sum','count'])
             .rename(columns={'sum':'pass','count':'total'})
             .assign(pass_rate=lambda d: (d['pass']/d['total']*100).round(1)))

In [ ]:
show_figure('03_quality_flags_summary.png', 'Quality Flag Summary')
show_figure('03_blur_score_distribution.png', 'Blur Score Distribution (Laplacian variance)')

---
## 6 · Identity Leakage Detection (Face Clustering)

In [ ]:
sweep = load_json(OUTPUTS / 'clustering_sweep.json')

if sweep:
    best_eps = sweep.get('best_eps', '—')
    lri      = sweep.get('leakage_risk_index', {}).get('lri', '—')
    verdict  = '🟢 SAFE' if isinstance(lri, float) and lri <= 5 else ('🟠 AT_RISK' if isinstance(lri, float) and lri <= 15 else '🔴 HIGH_RISK')

    display(Markdown(f"""
| Metric | Value |
|--------|-------|
| Best DBSCAN eps | `{best_eps}` |
| Leakage Risk Index (LRI) | `{lri:.2f}%` |
| Leakage verdict | {verdict} |
"""))

    # Sweep table
    eps_results = sweep.get('eps_results', [])
    if eps_results:
        display(pd.DataFrame(eps_results).style.highlight_max(subset=['silhouette_score'], color='lightgreen'))

In [ ]:
cluster_df = load_csv(OUTPUTS / 'cluster_assignments.csv')

if not cluster_df.empty:
    multi = cluster_df[cluster_df['leakage_risk_flag']]
    display(Markdown(f"""
| | Count |
|-|-------|
| Total images | {len(cluster_df)} |
| Images with embeddings | {cluster_df['embedding_extracted'].sum()} |
| Multi-image clusters | {cluster_df[cluster_df['cluster_id']>=0]['cluster_id'].nunique()} |
| Images in multi-image clusters | {len(multi)} |
"""))

In [ ]:
show_figure('04_tsne_by_cluster.png',  't-SNE — coloured by cluster')
show_figure('04_tsne_by_severity.png', 't-SNE — coloured by severity')
show_figure('04_cluster_size_histogram.png', 'Cluster Size Histogram')
show_figure('04_cluster_dendrogram.png', 'Agglomerative Dendrogram (truncated)')

---
## 7 · Train / Val / Test Splits

In [ ]:
split_summary = load_json(OUTPUTS / 'splits' / 'split_summary.json')

if split_summary:
    splits_info = split_summary.get('splits', {})
    rows = []
    for split_name, info in splits_info.items():
        row = {'Split': split_name, 'Total': info.get('n_images', '—')}
        for cls, cnt in info.get('class_distribution', {}).items():
            row[cls] = cnt
        rows.append(row)
    if rows:
        display(pd.DataFrame(rows).set_index('Split'))

    verif = split_summary.get('verification', {})
    overall = verif.get('overall_pass', None)
    display(Markdown(f"**Split verification**: {'✅ All checks passed' if overall else '❌ Verification failed — check split_summary.json'}"))

In [ ]:
show_figure('05_split_class_distribution.png', 'Class Distribution per Split')

---
## 8 · Feasibility Report

In [ ]:
report_json = load_json(REPORTS / 'feasibility_report.json')

if report_json:
    verdict = report_json.get('verdict', '—')
    badge   = {'FEASIBLE': '🟢', 'MARGINAL': '🟡', 'NOT_FEASIBLE': '🔴'}.get(verdict, '⚪')
    display(Markdown(f'# {badge} Dataset Verdict: **{verdict}**'))

    gates = report_json.get('decision_gates', {})
    gate_rows = [{'Gate': k, 'Result': '✅ Pass' if v else '❌ Fail'} for k, v in gates.items()]
    display(pd.DataFrame(gate_rows))

    recs = report_json.get('recommendations', [])
    if recs:
        display(Markdown('### Phase 1 Recommendations'))
        for r in recs:
            display(Markdown(f'- {r}'))

In [ ]:
# Display the full markdown report
report_md_path = PROJECT_ROOT / 'reports' / 'phase0' / 'feasibility_report.md'
if report_md_path.exists():
    with open(report_md_path) as f:
        display(Markdown(f.read()))
else:
    print('⚠️  feasibility_report.md not found — run step 06 first.')